<a href="https://colab.research.google.com/github/greek-nlp/benchmark/blob/main/pos_tagging_colab_monte_carlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Greek POS Tagging in Colab

This notebook is a dedicated Colab entrypoint for the Greek POS tagging benchmark in this repository.

Current state:
- The notebook sets up Ollama inside Colab.
- It benchmarks POS tagging on the repo's `ProkopidisUdDt` dataset loader.
- It repeats runs multiple times and reports mean and SEM, following the same Monte Carlo setup style as the suite notebook.


## Quick Start

Run the notebook top to bottom:
1. Setup cells
2. Benchmark configuration
3. Model pull cell
4. Task overview
5. Benchmark execution and results cells

Important: in Ollama, use `llama3.1:8b`, not `llama3.1:8b-instruct`.


## Repo Setup

Open this notebook from the repository in Colab, or clone/upload the repository files into the runtime before continuing.


## 1. Environment Setup


In [ ]:
!apt-get -qq update
!apt-get -qq install -y zstd


In [ ]:
!pip -q install pandas pywer openpyxl datasets scikit-learn matplotlib seaborn conll-df zenodo-get


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)
print('Started Ollama server with PID', ollama_process.pid)


## 2. Benchmark Configuration


In [ ]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd()
print('Working directory:', project_root)
print('Python executable:', sys.executable)

MODEL_PRESETS = {
    't4': [
        'qwen2.5:3b',
        'gemma2:2b',
        'mistral:7b',
        'qwen2.5:7b-instruct',
    ],
    'high_memory': [
        'llama3.1:8b',
        'aya-expanse:8b',
        'gemma2:9b',
        'qwen2.5:14b',
    ],
}

TASK_INFO = {
    'task': 'pos_tagging',
    'dataset': 'ProkopidisUdDt / UD Greek GDT',
    'label_column': 'x',
    'split': 'test',
}

MODEL_PRESET = 't4'
MODELS = MODEL_PRESETS[MODEL_PRESET]

SAMPLE_SIZE = 100
NUM_REPEATS = 5
RANDOM_STATE = 42
RESUME_REPEATS = True
OUTPUT_ROOT = Path('results/pos_tagging_colab')

print('Model preset:', MODEL_PRESET)
print('Models:', MODELS)
print('Task:', TASK_INFO['task'])
print('Dataset:', TASK_INFO['dataset'])
print('Label column:', TASK_INFO['label_column'])
print('Sample size:', SAMPLE_SIZE)
print('Repeats per task:', NUM_REPEATS)
print('Resume existing repeats:', RESUME_REPEATS)


In [ ]:
for model in MODELS:
    print(f'Pulling {model}...')
    subprocess.run(['ollama', 'pull', model], check=True)


In [ ]:
!ollama list


## 3. Task Overview


In [ ]:
pd.DataFrame([TASK_INFO])


## 4. Run POS Tagging


In [ ]:
import os

repo_url = 'https://github.com/greek-nlp/benchmark.git'
repo_dir = 'benchmark'

if not os.path.exists(repo_dir):
    print(f'Cloning {repo_url} into {repo_dir}...')
    !git clone {repo_url}
    %cd {repo_dir}
    print(f'Changed current working directory to {os.getcwd()}')
else:
    print(f'Repository {repo_dir} already exists. Pulling latest changes...')
    %cd {repo_dir}
    !git pull
    print(f'Current working directory is {os.getcwd()}')


In [ ]:
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from benchmark_suite import GenerationConfig, save_run_outputs
from benchmark_suite.backends import OllamaBackend


def aggregate_repeated_summaries(summary_by_repeat):
    repeated_summary = pd.concat(summary_by_repeat, ignore_index=True)
    key_columns = [column for column in ['task', 'model'] if column in repeated_summary.columns]
    numeric_columns = [
        column
        for column in repeated_summary.columns
        if column not in key_columns + ['repeat'] and pd.api.types.is_numeric_dtype(repeated_summary[column])
    ]

    aggregated = repeated_summary[key_columns].drop_duplicates().sort_values(key_columns).reset_index(drop=True)
    for column in numeric_columns:
        stats = (
            repeated_summary.groupby(key_columns, dropna=False)[column]
            .agg(['mean', 'sem'])
            .reset_index()
            .rename(columns={'mean': f'{column}_mean', 'sem': f'{column}_sem'})
        )
        aggregated = aggregated.merge(stats, on=key_columns, how='left')
    return aggregated, repeated_summary


def load_repeat_outputs(repeat_output_dir, task_name):
    summary = pd.read_csv(repeat_output_dir / f'{task_name}_summary.csv')
    raw = pd.read_csv(repeat_output_dir / f'{task_name}_predictions.csv')
    return summary, raw


gr_data = pd.read_csv('data.csv')
gena = importlib.import_module('data_wrapper')


def normalize_label(value):
    return ' '.join(str(value).split()).strip()


def prepare_dataset(random_state):
    prokopidis_ud = gena.ProkopidisUdDt(datasets=gr_data)
    token_df = prokopidis_ud.get(TASK_INFO['split']).copy()
    labels = sorted(token_df['x'].dropna().astype(str).map(normalize_label).unique())
    df = pd.DataFrame()
    df['tokens'] = token_df.groupby('s').w.apply(list)
    df['pos_tags'] = token_df.groupby('s').x.apply(lambda values: [normalize_label(v) for v in list(values)])
    df = df.reset_index(drop=True)
    if SAMPLE_SIZE and SAMPLE_SIZE < len(df):
        df = df.sample(SAMPLE_SIZE, random_state=random_state).reset_index(drop=True)
    return df, labels, TASK_INFO['split']


def build_prompt(example, pos_labels):
    return (
        'You are given a tokenized Greek sentence. Return only a Python-style list of POS tags, one tag per token.\n'
        f'Use only these labels: {pos_labels}\n\n'
        f'Tokens: {example["tokens"]}\n'
        'Tags:'
    )


def extract_list(text, labels):
    if not isinstance(text, str):
        return []
    text = text.strip().replace('"', '').replace("'", '').replace('\\n', ' ')
    text = text.replace('[', '').replace(']', '').replace(',', ' ')
    parsed = [normalize_label(item) for item in text.split()]
    return [item for item in parsed if item in labels]


def align_predictions(predicted_tags, gold_tags, filler):
    predicted_tags = predicted_tags[:len(gold_tags)]
    if len(predicted_tags) < len(gold_tags):
        predicted_tags = predicted_tags + [filler for _ in range(len(gold_tags) - len(predicted_tags))]
    return predicted_tags


def evaluate_predictions(raw, pos_labels):
    rows = []
    fallback_label = '_' if '_' in pos_labels else pos_labels[0]
    for model, group in raw.groupby('model', sort=False):
        gold = []
        predictions = []
        for row in group.itertuples(index=False):
            gold_tags = list(row.pos_tags)
            predicted_tags = align_predictions(list(row.prediction_tags), gold_tags, fallback_label)
            gold.extend(gold_tags)
            predictions.extend(predicted_tags)
        rows.append(
            {
                'task': TASK_INFO['task'],
                'model': model,
                'samples': len(group),
                'tokens': len(gold),
                'accuracy': accuracy_score(gold, predictions),
                'macro_f1': f1_score(gold, predictions, labels=pos_labels, average='macro', zero_division=0),
                'avg_latency_seconds': group['latency_seconds'].mean(),
            }
        )
    return pd.DataFrame(rows).sort_values(['macro_f1', 'accuracy'], ascending=[False, False]).reset_index(drop=True)


def run_task(models, random_state):
    dataset, pos_labels, split_name = prepare_dataset(random_state)
    backend = OllamaBackend()
    config = GenerationConfig(temperature=0.0, num_predict=256, timeout_seconds=300)

    records = []
    for model in models:
        for row in dataset.itertuples(index=False):
            example = row._asdict()
            result = backend.generate(
                model=model,
                prompt=build_prompt(example, pos_labels),
                system_prompt=(
                    'You are a Greek POS tagging system. '
                    'Return only a Python-style list of tags and nothing else.'
                ),
                temperature=config.temperature,
                num_predict=config.num_predict,
                timeout_seconds=config.timeout_seconds,
            )
            prediction_tags = extract_list(result.response, pos_labels)
            records.append(
                {
                    'task': TASK_INFO['task'],
                    'model': model,
                    'prediction': result.response,
                    'prediction_tags': prediction_tags,
                    'prediction_count': len(prediction_tags),
                    'latency_seconds': result.latency_seconds,
                    **example,
                }
            )

    raw = pd.DataFrame.from_records(records)
    summary = evaluate_predictions(raw, pos_labels)
    return summary, raw, split_name


In [ ]:
task_name = TASK_INFO['task']
repeat_summaries = []
repeat_predictions = []
split_names = []

for repeat_index in range(NUM_REPEATS):
    repeat_number = repeat_index + 1
    repeat_seed = RANDOM_STATE + repeat_index
    repeat_output_dir = OUTPUT_ROOT / task_name / f'repeat_{repeat_number:02d}'

    if RESUME_REPEATS and (repeat_output_dir / f'{task_name}_summary.csv').exists():
        print(f'Reusing repeat {repeat_number}/{NUM_REPEATS} from {repeat_output_dir}')
        summary, raw = load_repeat_outputs(repeat_output_dir, task_name)
        split_name = 'cached'
    else:
        print(f'Running repeat {repeat_number}/{NUM_REPEATS} with seed {repeat_seed}')
        summary, raw, split_name = run_task(MODELS, repeat_seed)
        repeat_output_dir.mkdir(parents=True, exist_ok=True)
        save_run_outputs(summary, raw, repeat_output_dir, task_name)

    summary = summary.copy()
    raw = raw.copy()
    summary['repeat'] = repeat_number
    raw['repeat'] = repeat_number
    repeat_summaries.append(summary)
    repeat_predictions.append(raw)
    split_names.append(split_name)

aggregated_summary, repeated_summary = aggregate_repeated_summaries(repeat_summaries)
repeated_predictions = pd.concat(repeat_predictions, ignore_index=True)

task_output_dir = OUTPUT_ROOT / task_name
task_output_dir.mkdir(parents=True, exist_ok=True)
aggregated_summary.to_csv(task_output_dir / f'{task_name}_summary_with_sem.csv', index=False)
repeated_summary.to_csv(task_output_dir / f'{task_name}_repeat_summaries.csv', index=False)
repeated_predictions.to_csv(task_output_dir / f'{task_name}_repeat_predictions.csv', index=False)

display(aggregated_summary)
print('Observed dataset split names:', split_names)
print(f'Saved aggregated outputs to {task_output_dir}')


In [ ]:
display(repeated_summary)

metric = 'macro_f1_mean' if 'macro_f1_mean' in aggregated_summary.columns else 'accuracy_mean'
sem_column = metric.replace('_mean', '_sem')
ranked = aggregated_summary.sort_values(metric, ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 6))
plt.bar(ranked['model'], ranked[metric], yerr=ranked[sem_column].fillna(0.0), capsize=6)
plt.ylabel(metric.replace('_mean', '').replace('_', ' ').title())
plt.xlabel('Model')
plt.title(f"{TASK_INFO['task'].replace('_', ' ').title()} ({metric.replace('_mean', '').replace('_', ' ').title()} mean +/- SEM)")
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
!zip pos_tagging_colab.zip -r results/pos_tagging_colab
